# Block 1 — confirm you can reach the device and see what the API responds to.

In [1]:
import requests

HOST = "10.15.22.19"

r = requests.get(f"http://{HOST}/", timeout=5)
print(r.status_code, len(r.text))

200 1114


### Reachable. 1114 chars is just the single-page-app shell, which is expected.

# Block 2 — probe for the API base path. We don't have docs yet, so let's see what responds:

In [2]:
paths = [
    "/api",
    "/api/v1",
    "/api/v1/real_time",
    "/api/v0/real_time",
    "/real_time",
    "/api/v1/tables",
]

for p in paths:
    try:
        r = requests.get(f"http://{HOST}{p}", timeout=5)
        print(f"{r.status_code}  {p}  {r.headers.get('content-type','')}  {r.text[:120]}")
    except Exception as e:
        print(f"ERR   {p}  {e}")

404  /api    
404  /api/v1    
404  /api/v1/real_time    
501  /api/v0/real_time    {"data":{},"meta":{"asset_id":"c0fe9f9d-439e-435a-a1c6-36ea56234a7a","data_version":"1.6","device_id":"3C:E4:B0:53:9D:EC
404  /real_time  text/html  <html>
<head><title>404 Not Found</title></head>
<body>
<center><h1>404 Not Found</h1></center>
<hr><center>nginx</c
404  /api/v1/tables    


In [3]:
import json

r = requests.get(f"http://{HOST}/api/v0/real_time", timeout=5)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:2000])

501
{
  "data": {},
  "meta": {
    "asset_id": "c0fe9f9d-439e-435a-a1c6-36ea56234a7a",
    "data_version": "1.6",
    "device_id": "3C:E4:B0:53:9D:EC",
    "error": {
      "detail": "Use /api/v0/real_time/{data,schema}",
      "status": "Not Implemented"
    }
  }
}


In [4]:
r = requests.get(f"http://{HOST}/api/v0/real_time/schema", timeout=10)
print(r.status_code)
s = r.json()
print(json.dumps(s, indent=2)[:3000])

200
{
  "data": {
    "categories": [
      {
        "name": "calendar_day",
        "subtype": "category_value",
        "type": "string"
      },
      {
        "name": "hour",
        "subtype": "category_value",
        "type": "string"
      },
      {
        "name": "information_source",
        "subtype": "category_value",
        "type": "string"
      },
      {
        "name": "job",
        "subtype": "category_value",
        "type": "string"
      },
      {
        "name": "month",
        "subtype": "category_value",
        "type": "string"
      },
      {
        "name": "ordinal_shift",
        "subtype": "category_value",
        "type": "string"
      },
      {
        "name": "part",
        "subtype": "category_value",
        "type": "string"
      },
      {
        "name": "performance_impact",
        "subtype": "category_value",
        "type": "string"
      },
      {
        "name": "process_state",
        "subtype": "category_value",
        "type":

In [5]:
import pandas as pd

d = r.json()["data"]
print("sections:", list(d.keys()))

for k, v in d.items():
    print(f"\n{k}: {len(v)}")
    print(pd.DataFrame(v).to_string())

sections: ['categories', 'derived_metrics', 'event_ids', 'metrics', 'metrics[channels]']

categories: 18
                    name         subtype    type
0           calendar_day  category_value  string
1                   hour  category_value  string
2     information_source  category_value  string
3                    job  category_value  string
4                  month  category_value  string
5          ordinal_shift  category_value  string
6                   part  category_value  string
7     performance_impact  category_value  string
8          process_state  category_value  string
9   process_state_reason  category_value  string
10        production_day  category_value  string
11      production_phase  category_value  string
12               quarter  category_value  string
13                 shift  category_value  string
14            shift_hour  category_value  string
15                  team  category_value  string
16                  week  category_value  string
17           

In [6]:
r = requests.get(f"http://{HOST}/api/v0/real_time/data", timeout=10)
print(r.status_code)
j = r.json()
print(list(j["data"].keys()))
print(json.dumps(j["data"], indent=2)[:1500])

200
['categories', 'category_display_names', 'derived_metrics', 'event_ids', 'metrics', 'pin_state']
{
  "categories": {
    "calendar_day": "4",
    "hour": "8",
    "information_source": "barcode_scan",
    "job": "3044003",
    "month": "10",
    "ordinal_shift": "1",
    "part": "ESTRADIOL",
    "performance_impact": "planned_stop",
    "process_state": "changeover",
    "process_state_reason": "changeover",
    "production_day": "4",
    "production_phase": "changeover",
    "quarter": "4",
    "shift": "first_shift",
    "shift_hour": "2",
    "team": "null",
    "week": "40",
    "year": "2026"
  },
  "category_display_names": {
    "calendar_day": "Friday",
    "hour": "8 AM",
    "information_source": "Barcode Scan",
    "job": "3044003",
    "month": "October",
    "ordinal_shift": "1",
    "part": "ESTRADIOL",
    "performance_impact": "Planned Stop",
    "process_state": "Changeover",
    "process_state_reason": "Changeover",
    "production_day": "Friday",
    "production_

In [7]:
d = j["data"]
channels = ["shift_hour", "shift", "part"]

rows = []
for ch in channels:
    row = {"channel": ch}
    row.update(d["categories"])
    for m, v in d["metrics"].items():
        if isinstance(v, dict):
            row[m] = v.get(ch)
    rows.append(row)

df = pd.DataFrame(rows)
print(df.shape)
print(df[["channel","part","shift","shift_hour","good_count","reject_count","run_time","unplanned_stop_time"]].to_string())

(3, 77)
      channel       part        shift shift_hour  good_count  reject_count       run_time  unplanned_stop_time
0  shift_hour  ESTRADIOL  first_shift          2         0.0           0.0       0.000000             0.000000
1       shift  ESTRADIOL  first_shift          2         0.0           0.0       0.000000           439.000000
2        part  ESTRADIOL  first_shift          2    225968.0      -91950.0  127413.666123         65181.625178


In [8]:
for p in ["/api/v0", "/api/v0/schema", "/api/v0/tables", "/api/v0/channels",
          "/api/v0/data", "/api/v0/events", "/api/v0/table"]:
    try:
        r = requests.get(f"http://{HOST}{p}", timeout=5)
        print(r.status_code, p, r.text[:200].replace("\n"," "))
    except Exception as e:
        print("ERR", p, e)

404 /api/v0 
404 /api/v0/schema 
200 /api/v0/tables {"data":{"tables":["production_metric","instant","quality_event","diagnostic","system","quality","notes","calendar_day_values","diagnostic_event_type_values","error_category_values","hour_values","inf
200 /api/v0/channels {"data":{"channels":["production_metric","process_state","instant","production_phase","calendar_day","shift","team","part","job","quality_event","diagnostic","hour","shift_hour","production_day","week
404 /api/v0/data 
404 /api/v0/events 
404 /api/v0/table 


In [9]:
for p in ["tables", "channels"]:
    r = requests.get(f"http://{HOST}/api/v0/{p}", timeout=10)
    items = list(r.json()["data"].values())[0]
    print(f"\n{p}: {len(items)}")
    for i in items:
        print(" ", i)


tables: 31
  production_metric
  instant
  quality_event
  diagnostic
  system
  quality
  notes
  calendar_day_values
  diagnostic_event_type_values
  error_category_values
  hour_values
  information_source_values
  internal_storage_lifetime_values
  job_values
  minute_values
  month_values
  ordinal_shift_values
  part_values
  performance_impact_values
  process_state_values
  process_state_reason_values
  production_day_values
  production_phase_values
  quarter_values
  reject_reason_values
  scoreboard_model_values
  shift_values
  shift_hour_values
  team_values
  week_values
  year_values

channels: 20
  production_metric
  process_state
  instant
  production_phase
  calendar_day
  shift
  team
  part
  job
  quality_event
  diagnostic
  hour
  shift_hour
  production_day
  week
  month
  quarter
  year
  system
  quality


In [10]:
for p in ["channels/shift_hour", "channels/process_state", "tables/production_metric"]:
    r = requests.get(f"http://{HOST}/api/v0/{p}", timeout=10)
    print(r.status_code, p)
    print(json.dumps(r.json(), indent=2)[:800])
    print("-"*60)

200 channels/shift_hour
{
  "data": {
    "fields": [
      {
        "display_name": "term.job_display_name.name",
        "help_text": "term.job_display_name.short",
        "name": "job_display_name",
        "scoreboard_name": "term.job_display_name.scoreboard_name",
        "subtype": "annotation",
        "type": "string"
      },
      {
        "display_name": "term.part_display_name.name",
        "help_text": "term.part_display_name.short",
        "name": "part_display_name",
        "scoreboard_name": "term.part_display_name.scoreboard_name",
        "subtype": "annotation",
        "type": "string"
      },
      {
        "display_name": "term.process_state_reason_display_name.name",
        "help_text": "term.process_state_reason_display_name.short",
        "name": "process_state_reason_display_na
------------------------------------------------------------
200 channels/process_state
{
  "data": {
    "fields": [
      {
        "display_name": "term.job_display_name.na

In [11]:
tests = [
    "channels/shift_hour/data",
    "channels/shift_hour/schema",
    "tables/production_metric/data",
    "tables/production_metric/rows",
]
for p in tests:
    r = requests.get(f"http://{HOST}/api/v0/{p}", timeout=10)
    print(r.status_code, p, "|", r.text[:300].replace("\n"," "))
    print("-"*60)

400 channels/shift_hour/data | {"data":{},"meta":{"asset_id":"c0fe9f9d-439e-435a-a1c6-36ea56234a7a","data_version":"1.6","device_id":"3C:E4:B0:53:9D:EC","error":{"detail":"Internal error","status":"Bad Request"}}}
------------------------------------------------------------
400 channels/shift_hour/schema | {"data":{},"meta":{"asset_id":"c0fe9f9d-439e-435a-a1c6-36ea56234a7a","data_version":"1.6","device_id":"3C:E4:B0:53:9D:EC","error":{"detail":"Internal error","status":"Bad Request"}}}
------------------------------------------------------------
404 tables/production_metric/data | {"error":"Not a valid tables request","success":false}
------------------------------------------------------------
404 tables/production_metric/rows | {"error":"Not a valid tables request","success":false}
------------------------------------------------------------


In [12]:
tests = [
    ("channels/shift_hour/data", {"fields": "good_count,reject_count,run_time"}),
    ("channels/shift_hour/data", {"limit": 5}),
    ("channels/shift_hour/data", {"count": 5}),
    ("channels/shift_hour/data", {"fields": "good_count", "limit": 5}),
]
for p, params in tests:
    r = requests.get(f"http://{HOST}/api/v0/{p}", params=params, timeout=10)
    print(r.status_code, params, "|", r.text[:300].replace("\n"," "))
    print("-"*60)

400 {'fields': 'good_count,reject_count,run_time'} | {"data":{},"meta":{"asset_id":"c0fe9f9d-439e-435a-a1c6-36ea56234a7a","data_version":"1.6","device_id":"3C:E4:B0:53:9D:EC","error":{"detail":"Internal error","status":"Bad Request"}}}
------------------------------------------------------------
400 {'limit': 5} | {"data":{},"meta":{"asset_id":"c0fe9f9d-439e-435a-a1c6-36ea56234a7a","data_version":"1.6","device_id":"3C:E4:B0:53:9D:EC","error":{"detail":"Internal error","status":"Bad Request"}}}
------------------------------------------------------------
400 {'count': 5} | {"data":{},"meta":{"asset_id":"c0fe9f9d-439e-435a-a1c6-36ea56234a7a","data_version":"1.6","device_id":"3C:E4:B0:53:9D:EC","error":{"detail":"Internal error","status":"Bad Request"}}}
------------------------------------------------------------
400 {'fields': 'good_count', 'limit': 5} | {"data":{},"meta":{"asset_id":"c0fe9f9d-439e-435a-a1c6-36ea56234a7a","data_version":"1.6","device_id":"3C:E4:B0:53:9D:EC","error":{"de

In [13]:
params = {
    "fields": "start_time,end_time,shift,shift_hour,part,good_count,reject_count,run_time,unplanned_stop_time",
    "sort": "-record_order",
}
r = requests.get(f"http://{HOST}/api/v0/channels/shift_hour/events",
                 params=params, timeout=15)
print(r.status_code)
print(json.dumps(r.json(), indent=2)[:1200])

200
{
  "data": {
    "events": [
      [
        "2026-10-02T09:00:00.000000-04:00",
        "2026-10-02T09:47:22.970334-04:00",
        "first_shift",
        "3",
        "ESTRADIOL",
        0.0,
        0.0,
        0.0,
        0.0
      ],
      [
        "2026-10-02T08:00:00.000000-04:00",
        "2026-10-02T09:00:00.000000-04:00",
        "first_shift",
        "2",
        "ESTRADIOL",
        0.0,
        0.0,
        0.0,
        0.0
      ],
      [
        "2026-10-02T07:00:00.000000-04:00",
        "2026-10-02T08:00:00.000000-04:00",
        "first_shift",
        "1",
        "ESTRADIOL",
        0.0,
        0.0,
        0.0,
        439.0
      ],
      [
        "2026-10-01T23:00:00.000000-04:00",
        "2026-10-02T07:00:00.000000-04:00",
        "null",
        "null",
        "ESTRADIOL",
        0.0,
        0.0,
        0.0,
        0.0
      ],
      [
        "2026-10-01T22:00:00.000000-04:00",
        "2026-10-01T23:00:00.000000-04:00",
        "second_shif

In [14]:
def get_events(channel, fields, sort="-record_order", filter=None, host=HOST):
    params = {"fields": ",".join(fields), "sort": sort}
    if filter:
        params["filter"] = filter
    r = requests.get(f"http://{host}/api/v0/channels/{channel}/events",
                     params=params, timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()["data"]["events"], columns=fields)

FIELDS = ["start_time","end_time","shift","shift_hour","part",
          "good_count","reject_count","run_time","unplanned_stop_time",
          "planned_stop_time","not_scheduled_time","ideal_time","target_count"]

df = get_events("shift_hour", FIELDS)
print(df.shape)
print(df.head(10).to_string())
print(df["start_time"].min(), "→", df["start_time"].max())

(3616, 13)
                         start_time                          end_time         shift shift_hour       part  good_count  reject_count    run_time  unplanned_stop_time  planned_stop_time  not_scheduled_time  ideal_time  target_count
0  2026-10-02T09:00:00.000000-04:00  2026-10-02T09:47:57.914477-04:00   first_shift          3  ESTRADIOL         0.0           0.0    0.000000             0.000000        2877.914477                 0.0       0.000      0.000000
1  2026-10-02T08:00:00.000000-04:00  2026-10-02T09:00:00.000000-04:00   first_shift          2  ESTRADIOL         0.0           0.0    0.000000             0.000000        3600.000000                 0.0       0.000      0.000000
2  2026-10-02T07:00:00.000000-04:00  2026-10-02T08:00:00.000000-04:00   first_shift          1  ESTRADIOL         0.0           0.0    0.000000           439.000000        3161.000000                 0.0       0.000    256.083334
3  2026-10-01T23:00:00.000000-04:00  2026-10-02T07:00:00.000000-04:00

In [15]:
df["start_time"] = pd.to_datetime(df["start_time"], utc=True)
df["end_time"]   = pd.to_datetime(df["end_time"], utc=True)
df["dur_s"] = (df["end_time"] - df["start_time"]).dt.total_seconds()

num = ["good_count","reject_count","run_time","unplanned_stop_time",
       "planned_stop_time","not_scheduled_time","ideal_time","target_count"]
df[num] = df[num].astype(float)

print("rows:", len(df), "| dupes on start_time:", df["start_time"].duplicated().sum())
print("\nnegative rejects:", (df.reject_count < 0).sum())
print("rows with any production:", (df.good_count > 0).sum())
print("\nduration distribution (s):")
print(df.dur_s.value_counts().head())
print("\ntime accounting check (should be ~0):")
acct = df.dur_s - (df.run_time + df.unplanned_stop_time + df.planned_stop_time + df.not_scheduled_time)
print(acct.describe())

rows: 3616 | dupes on start_time: 0

negative rejects: 905
rows with any production: 1526

duration distribution (s):
dur_s
3600.000000      3537
28800.000000       28
172800.000000      19
201600.000000       6
2877.914477         1
Name: count, dtype: int64

time accounting check (should be ~0):
count     3616.000000
mean         6.123015
std        374.932628
min       -173.468714
25%          0.000000
50%          0.000000
75%          0.000000
max      22544.641564
dtype: float64


In [16]:
df["acct_gap"] = acct

# are gaps confined to long rows?
print(df.groupby(df.dur_s > 3600)["acct_gap"].agg(["count","mean","min","max"]))

print("\n--- negative reject rows ---")
neg = df[df.reject_count < 0]
print("date range:", neg.start_time.min(), "→", neg.start_time.max())
print("by part:\n", neg.part.value_counts().head())
print("\nsample:")
print(neg[["start_time","part","good_count","reject_count","run_time"]].head(8).to_string())

       count        mean           min           max
dur_s                                               
False   3546   -0.114003 -1.734687e+02  4.547474e-13
True      70  322.072532 -3.637979e-12  2.254464e+04

--- negative reject rows ---
date range: 2026-03-12 17:00:00+00:00 → 2026-09-30 20:00:00+00:00
by part:
 part
ESTRADIOL        575
PREMARINi_30G     99
ADVIL             85
PREMARINi_4G      57
PREMARINi_14G     28
Name: count, dtype: int64

sample:
                  start_time       part  good_count  reject_count     run_time
27 2026-09-30 20:00:00+00:00  ESTRADIOL      2556.0       -2556.0     0.000000
28 2026-09-30 19:00:00+00:00  ESTRADIOL      1512.0       -1512.0     0.000000
29 2026-09-30 18:00:00+00:00  ESTRADIOL      4078.0       -1274.0  3250.406740
30 2026-09-30 17:00:00+00:00  ESTRADIOL      5783.0       -2643.0  3391.199019
31 2026-09-30 16:00:00+00:00  ESTRADIOL      3044.0       -1098.0  1597.570624
32 2026-09-30 15:00:00+00:00  ESTRADIOL      6112.0       -2790

In [17]:
# does it net out at part-run level?
chk = df[df.part == "ESTRADIOL"].copy()
print("ESTRADIOL totals:")
print(" good:  ", chk.good_count.sum())
print(" reject:", chk.reject_count.sum())
print(" total: ", chk.good_count.sum() + chk.reject_count.sum())

# isolate the signature: reject == -good AND run_time == 0
sig = df[(df.reject_count == -df.good_count) & (df.run_time == 0) & (df.good_count > 0)]
print(f"\npure adjustment rows: {len(sig)} of {(df.reject_count<0).sum()} negative rows")
print("their good_count sum:", sig.good_count.sum())

ESTRADIOL totals:
 good:   2420633.0
 reject: -824415.0
 total:  1596218.0

pure adjustment rows: 193 of 905 negative rows
their good_count sum: 126693.0


In [18]:
work = df[df.dur_s == 3600].copy()
work["availability"] = work.run_time / (work.run_time + work.unplanned_stop_time + work.planned_stop_time)
work["performance"]  = work.ideal_time / work.run_time

run = work[work.run_time > 0]
print(f"rows with run_time > 0: {len(run)}")
print(run[["availability","performance"]].describe().to_string())
print("\nperformance > 1 (bad ICT):", (run.performance > 1).sum())

rows with run_time > 0: 1273
       availability  performance
count   1273.000000  1273.000000
mean       0.669137     0.448864
std        0.332082     0.231644
min        0.000646     0.000000
25%        0.382824     0.296720
50%        0.804759     0.408194
75%        0.963837     0.588603
max        1.000000     1.024964

performance > 1 (bad ICT): 13


In [19]:
# documented filter syntax
r = requests.get(f"http://{HOST}/api/v0/channels/shift_hour/events",
    params={"fields":"start_time,good_count,reject_count",
            "production_day_number[gt]":"20700","limit":5}, timeout=15)
print(r.status_code, r.json()["data"]["events"][:3])

# reject-level detail
r2 = requests.get(f"http://{HOST}/api/v0/channels/quality_event",
                  params={}, timeout=15)
print([f["name"] for f in r2.json()["data"]["fields"]])

200 [['2026-09-05T07:00:00.000000-04:00', 0.0, 0.0], ['2026-09-07T07:00:00.000000-04:00', 0.0, 0.0], ['2026-09-07T08:00:00.000000-04:00', 0.0, 0.0]]
['reject_reason_display_name', 'count', 'event_id', 'production_day_number', 'record_id', 'record_order', 'slice_order', 'sync_id', 'reject_reason']


In [20]:
qf = ["production_day_number","reject_reason","reject_reason_display_name","count","record_order"]
q = get_events("quality_event", qf)
q["count"] = q["count"].astype(float)
print(q.shape)

print("\nnegative count rows:", (q["count"] < 0).sum())
print("\nby reason (sum / n / min):")
print(q.groupby("reject_reason_display_name")["count"]
       .agg(["sum","count","min"]).sort_values("sum").to_string())

(0, 5)

negative count rows: 0

by reason (sum / n / min):
Empty DataFrame
Columns: [sum, count, min]
Index: []


In [21]:
F = ["start_time","in_count","good_count","reject_count","pack_out_count","run_time"]
t = get_events("shift_hour", F)
for c in F[1:]:
    t[c] = t[c].astype(float)

t["implied"] = t.in_count - t.good_count
print(t[["in_count","good_count","reject_count","implied","pack_out_count"]].describe().to_string())
print("\nrows where reject == in - good:", (t.reject_count == t["implied"]).sum(), "of", len(t))
print("\nsample with production:")
print(t[t.good_count>0][["start_time","in_count","good_count","reject_count","implied"]].head(8).to_string())

           in_count    good_count  reject_count       implied  pack_out_count
count   3616.000000   3616.000000   3616.000000   3616.000000     3616.000000
mean     923.392423   1122.769635   -199.232854   -199.377212      108.869469
std     2160.983393   2968.802439   1390.798433   1391.864169      585.532226
min        0.000000  -4806.000000 -31896.000000 -31896.000000        0.000000
25%        0.000000      0.000000     -1.000000     -1.000000        0.000000
50%        0.000000      0.000000      0.000000      0.000000        0.000000
75%     1387.500000   1252.750000      0.000000      0.000000        0.000000
max    53048.000000  83036.000000  24276.000000  24276.000000     5616.000000

rows where reject == in - good: 3612 of 3616

sample with production:
                          start_time  in_count  good_count  reject_count  implied
10  2026-10-01T16:00:00.000000-04:00     948.0       477.0         471.0    471.0
11  2026-10-01T15:00:00.000000-04:00    2828.0      2162.0     

In [22]:
# does it net out over longer windows?
t["start_time"] = pd.to_datetime(t["start_time"], utc=True)
t["day"] = t["start_time"].dt.date

for grain, g in [("hour", t), ("day", t.groupby("day").sum(numeric_only=True).reset_index())]:
    bad = (g.reject_count < 0).sum()
    print(f"{grain:5} rows={len(g):5}  negative={bad:4} ({bad/len(g):.0%})  "
          f"net_reject={g.reject_count.sum():,.0f}")

# shift grain, straight from the API
s = get_events("shift", ["start_time","shift","in_count","good_count","reject_count"])
for c in ["in_count","good_count","reject_count"]:
    s[c] = s[c].astype(float)
print(f"shift rows={len(s)}  negative={(s.reject_count<0).sum()} "
      f"({(s.reject_count<0).mean():.0%})  net_reject={s.reject_count.sum():,.0f}")

hour  rows= 3616  negative= 905 (25%)  net_reject=-720,426
day   rows=  194  negative=  85 (44%)  net_reject=-720,426
shift rows=516  negative=178 (34%)  net_reject=-720,426


In [23]:
import concurrent.futures as cf

def probe(ip):
    try:
        r = requests.get(f"http://{ip}/api/v0/device", timeout=2)
        if r.status_code == 200:
            d = r.json()
            return ip, d.get("asset_name",{}).get("value"), d.get("firmware_version",{}).get("value")
    except Exception:
        return None

ips = [f"10.15.22.{i}" for i in range(1, 255)]
with cf.ThreadPoolExecutor(50) as ex:
    found = [x for x in ex.map(probe, ips) if x]

for f in found:
    print(f)

('10.15.22.19', None, None)
('10.15.22.83', None, None)
('10.15.22.95', None, None)
('10.15.22.119', None, None)


In [24]:
DEVICES = ["10.15.22.19", "10.15.22.83", "10.15.22.95", "10.15.22.119"]

for ip in DEVICES:
    d = requests.get(f"http://{ip}/api/v0/device", timeout=5).json()
    inner = d.get("data", d)
    name = inner.get("asset_name")
    fw   = inner.get("firmware_version")
    print(ip, "|", name, "|", fw)

10.15.22.19 | {'metadata': {'name_key': 'asset_name_literal', 'type': 'string'}, 'value': 'FR8'} | {'metadata': {'name_key': 'firmware_version_literal', 'type': 'string'}, 'value': '2.22.14.2'}
10.15.22.83 | {'metadata': {'name_key': 'asset_name_literal', 'type': 'string'}, 'value': 'AreaB'} | {'metadata': {'name_key': 'firmware_version_literal', 'type': 'string'}, 'value': '2.22.14.2'}
10.15.22.95 | {'metadata': {'name_key': 'asset_name_literal', 'type': 'string'}, 'value': 'AreaD'} | {'metadata': {'name_key': 'firmware_version_literal', 'type': 'string'}, 'value': '2.22.14.2'}
10.15.22.119 | {'metadata': {'name_key': 'asset_name_literal', 'type': 'string'}, 'value': 'AreaE'} | {'metadata': {'name_key': 'firmware_version_literal', 'type': 'string'}, 'value': '2.22.14.2'}


In [25]:
ASSETS = {"10.15.22.19":"FR8", "10.15.22.83":"AreaB",
          "10.15.22.95":"AreaD", "10.15.22.119":"AreaE"}

F = ["start_time","end_time","shift","shift_hour","part","job",
     "in_count","good_count","reject_count","run_time",
     "unplanned_stop_time","planned_stop_time","not_scheduled_time",
     "ideal_time","target_count"]

frames = []
for ip, name in ASSETS.items():
    d = get_events("shift_hour", F, host=ip)
    d.insert(0, "asset", name)
    frames.append(d)
    print(f"{name:6} {len(d):5} rows")

all_df = pd.concat(frames, ignore_index=True)

num = [c for c in F if c not in ("start_time","end_time","shift","shift_hour","part","job")]
all_df[num] = all_df[num].astype(float)
all_df["start_time"] = pd.to_datetime(all_df["start_time"], utc=True)
all_df["end_time"]   = pd.to_datetime(all_df["end_time"], utc=True)

print("\ntotal:", all_df.shape)
print(all_df.groupby("asset").agg(
    rows=("start_time","size"),
    first=("start_time","min"),
    neg_pct=("reject_count", lambda s: f"{(s<0).mean():.0%}"),
    net_reject=("reject_count","sum"),
).to_string())

FR8     3616 rows
AreaB   2888 rows
AreaD   3042 rows
AreaE   2944 rows

total: (12490, 16)
       rows                     first neg_pct  net_reject
asset                                                    
AreaB  2888 2026-03-25 12:00:00+00:00     14%    278582.0
AreaD  3042 2026-03-26 12:00:00+00:00     13%     51768.0
AreaE  2944 2026-04-01 12:00:00+00:00     13%    -20281.0
FR8    3616 2026-01-20 13:00:00+00:00     25%   -720426.0


In [26]:
w = all_df[(all_df.end_time - all_df.start_time).dt.total_seconds() == 3600].copy()
w["prod_time"] = w.run_time + w.unplanned_stop_time + w.planned_stop_time
w["date"] = w.start_time.dt.tz_convert("America/Toronto").dt.date

agg = w.groupby("asset").agg(
    hours=("run_time","size"),
    run=("run_time","sum"),
    unplanned=("unplanned_stop_time","sum"),
    planned=("planned_stop_time","sum"),
    notsched=("not_scheduled_time","sum"),
    ideal=("ideal_time","sum"),
)
agg["availability"] = agg.run / (agg.run + agg.unplanned + agg.planned)
agg["performance"]  = agg.ideal / agg.run
agg["utilization"]  = (agg.run + agg.unplanned + agg.planned) / (agg.run + agg.unplanned + agg.planned + agg.notsched)
print(agg[["hours","availability","performance","utilization"]].to_string(
    float_format=lambda x: f"{x:,.3f}"))

       hours  availability  performance  utilization
asset                                               
AreaB   2832         0.378        0.609        0.598
AreaD   2984         0.349        0.692        0.566
AreaE   2889         0.332        0.394        0.385
FR8     3537         0.330        0.489        0.660


In [27]:
from pathlib import Path

C = {"run":"#1F6F5C","unp":"#B3261E","pln":"#C98A0B","nsc":"#B9C0C4"}

tot = w.groupby("asset")[["run_time","unplanned_stop_time","planned_stop_time",
                          "not_scheduled_time","ideal_time"]].sum()
tot["all"] = tot.iloc[:,:4].sum(axis=1)
tot["avail"] = tot.run_time/(tot.run_time+tot.unplanned_stop_time+tot.planned_stop_time)
tot["perf"]  = tot.ideal_time/tot.run_time
tot["util"]  = (tot["all"]-tot.not_scheduled_time)/tot["all"]
tot = tot.sort_values("avail", ascending=False)

def bar(r):
    seg = [("run_time","run"),("unplanned_stop_time","unp"),
           ("planned_stop_time","pln"),("not_scheduled_time","nsc")]
    return "".join(f'<i style="width:{r[k]/r["all"]*100:.3f}%;background:{C[c]}"></i>'
                   for k,c in seg)

rows = "".join(f"""<tr><th>{a}</th><td class=bar>{bar(r)}</td>
<td>{r.avail:.1%}</td><td>{r.perf:.1%}</td><td>{r.util:.1%}</td>
<td>{r['all']/3600:,.0f}</td></tr>""" for a,r in tot.iterrows())

d = (w.groupby(["asset","date"])[["run_time","unplanned_stop_time","planned_stop_time"]]
      .sum().reset_index())
d["a"] = d.run_time/(d.run_time+d.unplanned_stop_time+d.planned_stop_time)
strips = ""
for a in tot.index:
    g = d[d.asset==a]
    cells = "".join(f'<i title="{x.date} {x.a:.0%}" style="height:{max(x.a*100,1.5):.1f}%"></i>'
                    for x in g.itertuples())
    strips += f'<div class=strip><h3>{a}</h3><div class=spark>{cells}</div></div>'

neg = all_df.groupby("asset").reject_count.agg(["sum", lambda s:(s<0).mean()])
integ = "".join(f"<tr><th>{a}</th><td>{v['sum']:,.0f}</td><td>{v['<lambda_0>']:.0%}</td></tr>"
                for a,v in neg.iterrows())

span = f"{w.start_time.min():%d %b %Y} – {w.start_time.max():%d %b %Y}"

html = f"""<!doctype html><meta charset=utf-8><title>Danbro packaging — time accounting</title>
<link href="https://fonts.googleapis.com/css2?family=Libre+Franklin:wght@400;600;800&display=swap" rel=stylesheet>
<style>
:root{{--ink:#15191B;--rule:#DADFE1;--panel:#F4F6F6}}
*{{box-sizing:border-box}}
body{{margin:0;background:#fff;color:var(--ink);
font:400 16px/1.5 'Libre Franklin',system-ui,sans-serif;font-variant-numeric:tabular-nums}}
.wrap{{max-width:1080px;margin:0 auto;padding:56px 24px 96px}}
h1{{font-weight:800;font-size:clamp(28px,4vw,44px);letter-spacing:-.02em;margin:0 0 4px}}
.sub{{color:#5A6368;margin:0 0 48px}}
h2{{font-size:20px;font-weight:600;margin:56px 0 4px;padding-bottom:8px;border-bottom:2px solid var(--ink)}}
.note{{color:#5A6368;font-size:14px;margin:8px 0 20px;max-width:68ch}}
table{{width:100%;border-collapse:collapse}}
th,td{{text-align:right;padding:11px 10px;border-bottom:1px solid var(--rule);font-weight:400}}
th{{text-align:left;font-weight:600;white-space:nowrap}}
thead td,thead th{{font-size:13px;color:#5A6368;border-bottom:1px solid var(--ink)}}
td.bar{{width:46%;padding:11px 10px}}
.bar i{{display:inline-block;height:18px;vertical-align:middle}}
.key{{display:flex;flex-wrap:wrap;gap:18px;margin:16px 0 0;font-size:13px;color:#5A6368}}
.key i{{display:inline-block;width:11px;height:11px;margin-right:6px}}
.strip{{margin:22px 0}}
.strip h3{{font-size:14px;font-weight:600;margin:0 0 6px}}
.spark{{display:flex;align-items:flex-end;gap:2px;height:52px;background:var(--panel);padding:3px}}
.spark i{{flex:1;background:#1F6F5C;min-width:2px}}
.flag{{background:var(--panel);border-left:3px solid #B3261E;padding:18px 20px;margin:20px 0}}
.flag p{{margin:0 0 10px;font-size:14px;max-width:68ch}}
footer{{margin-top:72px;padding-top:16px;border-top:1px solid var(--rule);font-size:13px;color:#5A6368}}
@media(max-width:640px){{td.bar{{display:none}}}}
</style>
<div class=wrap>
<h1>Danbro packaging</h1>
<p class=sub>Where the time went, {span} · four lines · hourly records from Vorne XL</p>

<h2>Time accounting</h2>
<p class=note>Every scheduled second lands in one of four buckets. These add up to elapsed time to
within rounding, which is what makes the rates below dependable.</p>
<table><thead><tr><th>Line</th><td>Share of elapsed time</td><td>Available</td>
<td>Performance</td><td>Utilised</td><td>Hours</td></tr></thead><tbody>{rows}</tbody></table>
<div class=key>
<span><i style="background:{C['run']}"></i>Running</span>
<span><i style="background:{C['unp']}"></i>Unplanned stop</span>
<span><i style="background:{C['pln']}"></i>Planned stop</span>
<span><i style="background:{C['nsc']}"></i>Not scheduled</span></div>

<h2>Availability, day by day</h2>
<p class=note>Each bar is one production day. Full height means the line ran for all the
time it was scheduled to run.</p>
{strips}

<h2>Piece counts are not yet reliable</h2>
<div class=flag>
<p>Reject count on these devices is derived as infeed minus good count, not from scanned reject
reasons — the reject-event log is empty across the whole history. Where the good-count sensor
registers more pieces than the infeed sensor saw, rejects go negative.</p>
<p>Aggregating does not resolve it: the net is unchanged at hour, shift and day level, so this is a
measurement fault rather than a timing artefact. Quality and OEE are therefore withheld until the
count inputs are reviewed.</p>
<table><thead><tr><th>Line</th><td>Net rejects</td><td>Hours with negative rejects</td></tr></thead>
<tbody>{integ}</tbody></table>
</div>

<footer>Pulled from four XL devices on software 2.22.14.2 via the channel API.
Availability = run ÷ scheduled. Performance = ideal ÷ run. Utilised = scheduled ÷ elapsed.</footer>
</div>"""

Path("vorne_report.html").write_text(html, encoding="utf-8")
print("wrote vorne_report.html", len(html), "bytes")

wrote vorne_report.html 38052 bytes


In [28]:
# what's available
sc = requests.get(f"http://{DEVICES[0]}/api/v0/channels/process_state", timeout=10).json()
names = [f["name"] for f in sc["data"]["fields"]]
print([n for n in names if "state" in n or "impact" in n or "phase" in n or "time" in n])

PS = ["start_time","end_time","duration","process_state","process_state_reason",
      "process_state_reason_display_name","performance_impact","production_phase",
      "part","shift","production_day_number"]

ps = []
for ip, name in ASSETS.items():
    d = get_events("process_state", PS, host=ip)
    d.insert(0, "asset", name)
    ps.append(d)
    print(f"{name:6} {len(d):6} events")

psdf = pd.concat(ps, ignore_index=True)
psdf["duration"] = psdf["duration"].astype(float)
psdf["start_time"] = pd.to_datetime(psdf["start_time"], utc=True)

print("\n", psdf.shape)
print(psdf.groupby(["process_state","performance_impact"])
        .agg(events=("duration","size"), hours=("duration", lambda s: s.sum()/3600))
        .to_string())

['process_state_reason_display_name', 'current_cycle_time', 'elapsed_time', 'end_time', 'ideal_cycle_time', 'ideal_time', 'last_down_time', 'last_not_scheduled_time', 'last_planned_stop_time', 'last_run_time', 'last_unplanned_stop_time', 'not_scheduled_time', 'pace_timer', 'partial_cycle_lost_time', 'planned_stop_time', 'previous_cycle_time', 'process_state_event_id', 'process_state_remaining_time', 'production_phase_event_id', 'production_phase_remaining_time', 'run_cycle_lost_time', 'run_time', 'small_stop_lost_time', 'start_time', 'takt_time', 'target_time', 'unplanned_stop_time', 'total_cycle_time', 'average_ideal_cycle_time', 'average_cycle_time', 'average_run_cycle_time', 'average_small_stop_time', 'manufacturing_time', 'production_time', 'production_lost_time', 'all_time', 'fully_productive_time', 'availability_lost_time', 'performance_count_lost_time', 'performance_lost_time', 'performance_cycle_lost_time', 'quality_lost_time', 'oee_lost_time', 'capacity_lost_time', 'cycle_lost

In [29]:
psdf["hours"] = psdf.duration / 3600
R = (psdf.groupby(["asset","process_state","performance_impact",
                   "process_state_reason_display_name"])
        .agg(hours=("hours","sum"), events=("hours","size"))
        .reset_index().sort_values("hours", ascending=False))

print("reasons per asset:", R.groupby("asset").size().to_dict())
print("\nFR8 top 12 unplanned:")
print(R[(R.asset=="FR8") & (R.performance_impact=="unplanned_stop")]
        .head(12)[["process_state_reason_display_name","hours","events"]].to_string(index=False))

reasons per asset: {'AreaB': 68, 'AreaD': 55, 'AreaE': 47, 'FR8': 81}

FR8 top 12 unplanned:
process_state_reason_display_name      hours  events
           End of Lot Reclamation 202.535364      46
                   Missing Reason  80.756017     132
                     Tube Feeding  77.612412      95
                    Pusher Safety  54.285415     224
               Bulk Replenishment  49.890243      42
              LeafletFeeder Fault  26.530316     232
                  Product Feeding  24.748845      21
                   Waiting - Bulk  23.172693       9
                    Tube Loading!  21.062622       7
              ABBRobot Mech Issue  20.205902       4
              Waiting Components!  18.684404       2
                  Shipper Loading  15.461282      23


In [30]:
import json as _json
from pathlib import Path

OUT = Path("report"); OUT.mkdir(exist_ok=True)
TZ = "America/Toronto"

e = psdf.copy()
e["hours"] = e.duration / 3600
e["d"] = e.start_time.dt.tz_convert(TZ).dt.strftime("%Y-%m-%d")
e = e[["asset","d","process_state","performance_impact",
       "process_state_reason_display_name","hours"]]
e.columns = ["asset","d","state","impact","reason","h"]

# collapse to day × state × reason so the payload stays small
slim = (e.groupby(["asset","d","state","impact","reason"], as_index=False)
          .agg(h=("h","sum"), n=("h","size")))

hr = w.copy()
hr["d"] = hr.start_time.dt.tz_convert(TZ).dt.strftime("%Y-%m-%d")
daily = (hr.groupby(["asset","d"], as_index=False)
           .agg(run=("run_time","sum"), unp=("unplanned_stop_time","sum"),
                pln=("planned_stop_time","sum"), nsc=("not_scheduled_time","sum"),
                ideal=("ideal_time","sum")))
for c in ["run","unp","pln","nsc","ideal"]:
    daily[c] = (daily[c]/3600).round(4)

payload = {"assets": sorted(ASSETS.values()),
           "daily": daily.to_dict("records"),
           "activity": slim.round(4).to_dict("records")}
(OUT/"data.json").write_text(_json.dumps(payload, separators=(",",":")))
print("data.json", (OUT/"data.json").stat().st_size//1024, "KB",
      "| daily", len(daily), "| activity", len(slim))

data.json 560 KB | daily 649 | activity 4323


In [31]:
import json as _json
from pathlib import Path

PAGE = r"""<!doctype html><html lang=en><meta charset=utf-8>
<meta name=viewport content="width=device-width,initial-scale=1">
<title>Danbro packaging</title>
<link href="https://fonts.googleapis.com/css2?family=Libre+Franklin:wght@400;600;800&display=swap" rel=stylesheet>
<style>
:root{--ink:#15191B;--mute:#5A6368;--rule:#DADFE1;--panel:#F4F6F6;
--run:#1F6F5C;--unp:#B3261E;--pln:#C98A0B;--nsc:#B9C0C4}
*{box-sizing:border-box}
body{margin:0;background:#fff;color:var(--ink);
font:400 16px/1.5 'Libre Franklin',system-ui,sans-serif;font-variant-numeric:tabular-nums}
.wrap{max-width:1080px;margin:0 auto;padding:48px 24px 96px}
h1{font-weight:800;font-size:clamp(26px,4vw,40px);letter-spacing:-.02em;margin:0 0 4px}
.sub{color:var(--mute);margin:0 0 28px}
nav{display:flex;gap:4px;flex-wrap:wrap;border-bottom:1px solid var(--rule);margin-bottom:24px}
nav button{background:none;border:0;border-bottom:2px solid transparent;margin-bottom:-1px;
padding:9px 12px;font:inherit;font-weight:600;color:var(--mute);cursor:pointer}
nav button[aria-current=page]{color:var(--ink);border-bottom-color:var(--ink)}
nav button:focus-visible{outline:2px solid var(--ink);outline-offset:2px}
.ctrl{display:flex;gap:14px;align-items:center;flex-wrap:wrap;background:var(--panel);
padding:12px 14px;margin-bottom:8px}
.seg{display:flex;border:1px solid var(--rule);background:#fff}
.seg button{background:none;border:0;padding:7px 13px;font:inherit;font-size:14px;cursor:pointer;color:var(--mute)}
.seg button[aria-pressed=true]{background:var(--ink);color:#fff}
select{font:inherit;font-size:14px;padding:7px 9px;border:1px solid var(--rule);background:#fff;color:inherit}
.ctrl span{font-size:13px;color:var(--mute)}
h2{font-size:19px;font-weight:600;margin:44px 0 4px;padding-bottom:8px;border-bottom:2px solid var(--ink)}
.note{color:var(--mute);font-size:14px;margin:8px 0 18px;max-width:68ch}
table{width:100%;border-collapse:collapse}
th,td{text-align:right;padding:10px;border-bottom:1px solid var(--rule);font-weight:400}
th{text-align:left;font-weight:600}
thead th,thead td{font-size:13px;color:var(--mute);border-bottom:1px solid var(--ink);font-weight:600}
td.bar,th.bar{width:40%}
.bar i{display:inline-block;height:17px;vertical-align:middle}
.key{display:flex;flex-wrap:wrap;gap:18px;margin-top:14px;font-size:13px;color:var(--mute)}
.key i{display:inline-block;width:11px;height:11px;margin-right:6px}
.strip{margin:20px 0}
.strip h3{font-size:14px;font-weight:600;margin:0 0 6px}
.spark{display:flex;align-items:flex-end;gap:2px;height:48px;background:var(--panel);padding:3px}
.spark i{flex:1;background:var(--run);min-width:2px}
.flag{background:var(--panel);border-left:3px solid var(--unp);padding:16px 18px;margin:18px 0}
.flag p{margin:0 0 10px;font-size:14px;max-width:68ch}
.empty{color:var(--mute);padding:28px 0}
footer{margin-top:72px;padding-top:16px;border-top:1px solid var(--rule);font-size:13px;color:var(--mute)}
@media(max-width:700px){td.bar,th.bar{display:none}}
</style>
<div class=wrap>
<h1>Danbro packaging</h1>
<p class=sub id=sub></p>
<nav id=nav></nav>
<div class=ctrl>
<div class=seg id=gran></div>
<select id=period></select>
<span id=range></span>
</div>
<main id=main></main>
<footer>Four Vorne XL devices on software 2.22.14.2, pulled via the channel API.
Available = running ÷ scheduled. Performance = ideal ÷ running. Utilised = scheduled ÷ elapsed.
Piece counts are excluded; see the note on the overview.</footer>
</div>
<script>
const DATA=__DATA__, D=DATA.daily, A=DATA.activity, ASSETS=DATA.assets;
const IMP={run:'run',unplanned_stop:'unp',planned_stop:'pln',not_scheduled:'nsc'};
const MON=['January','February','March','April','May','June','July','August','September','October','November','December'];
const dates=[...new Set(D.map(r=>r.d))].sort();
let gran='month', period=null, page='overview';

const mk=d=>d.slice(0,7);
function wk(d){const t=new Date(d+'T00:00:00');t.setDate(t.getDate()-((t.getDay()+6)%7));return t.toISOString().slice(0,10)}
const nice=d=>{const[y,m,dd]=d.split('-');return +dd+' '+MON[+m-1].slice(0,3)+' '+y};
function periodList(g){
  if(g==='ytd')return [dates[dates.length-1].slice(0,4)];
  if(g==='day')return [...dates].reverse();
  const f=g==='month'?mk:wk;
  return [...new Set(dates.map(f))].sort().reverse();
}
function label(g,p){
  if(g==='ytd')return p+' to date';
  if(g==='day')return nice(p);
  if(g==='month')return MON[+p.slice(5,7)-1]+' '+p.slice(0,4);
  return 'Week of '+nice(p);
}
const inP=d=>gran==='ytd'?d.slice(0,4)===period:gran==='day'?d===period:(gran==='month'?mk:wk)(d)===period;

function agg(rs){const s={run:0,unp:0,pln:0,nsc:0,ideal:0};
  rs.forEach(r=>{s.run+=r.run;s.unp+=r.unp;s.pln+=r.pln;s.nsc+=r.nsc;s.ideal+=r.ideal});
  s.sched=s.run+s.unp+s.pln;s.all=s.sched+s.nsc;
  s.avail=s.sched?s.run/s.sched:0;s.perf=s.run?s.ideal/s.run:0;s.util=s.all?s.sched/s.all:0;return s}
const pc=v=>v?(v*100).toFixed(1)+'%':'—';
const hrs=v=>v>=100?Math.round(v).toLocaleString():v.toFixed(1);
function stack(s){const t=s.all||1;
  return [['run',s.run],['unp',s.unp],['pln',s.pln],['nsc',s.nsc]]
    .map(([k,v])=>v<=0?'':`<i style="width:${(v/t*100).toFixed(3)}%;background:var(--${k})" title="${hrs(v)} h"></i>`).join('')}
const KEY=`<div class=key><span><i style="background:var(--run)"></i>Running</span>
<span><i style="background:var(--unp)"></i>Unplanned stop</span>
<span><i style="background:var(--pln)"></i>Planned stop</span>
<span><i style="background:var(--nsc)"></i>Not scheduled</span></div>`;

function strips(list){return list.map(a=>{
  const g=D.filter(r=>r.asset===a&&inP(r.d)).sort((x,y)=>x.d<y.d?-1:1);
  if(!g.length)return '';
  const c=g.map(r=>{const s=r.run+r.unp+r.pln,v=s?r.run/s:0;
    return `<i title="${nice(r.d)} · ${pc(v)}" style="height:${Math.max(v*100,1.5).toFixed(1)}%"></i>`}).join('');
  return `<div class=strip><h3>${a}</h3><div class=spark>${c}</div></div>`}).join('')}

function overview(){
  const rows=ASSETS.map(a=>({a,s:agg(D.filter(r=>r.asset===a&&inP(r.d)))}))
    .filter(x=>x.s.all>0).sort((x,y)=>y.s.avail-x.s.avail);
  if(!rows.length)return '<p class=empty>No records in this period.</p>';
  const body=rows.map(({a,s})=>`<tr><th><button class=lnk data-go="${a}">${a}</button></th>
<td class=bar>${stack(s)}</td><td>${pc(s.avail)}</td><td>${pc(s.perf)}</td>
<td>${pc(s.util)}</td><td>${hrs(s.sched)}</td><td>${hrs(s.unp)}</td></tr>`).join('');
  const t=agg(D.filter(r=>inP(r.d)));
  return `<h2>Time accounting</h2>
<p class=note>Every second lands in one of four buckets, and they add to elapsed time within rounding.
That reconciliation is what makes the rates below dependable.</p>
<table><thead><tr><th>Line</th><td class=bar>Share of elapsed time</td><td>Available</td>
<td>Performance</td><td>Utilised</td><td>Scheduled h</td><td>Down h</td></tr></thead>
<tbody>${body}</tbody>
<tfoot><tr><th>All four</th><td class=bar>${stack(t)}</td><td>${pc(t.avail)}</td>
<td>${pc(t.perf)}</td><td>${pc(t.util)}</td><td>${hrs(t.sched)}</td><td>${hrs(t.unp)}</td></tr></tfoot>
</table>${KEY}
<h2>Availability, day by day</h2>
<p class=note>One bar per production day. Full height means the line ran for all the time it was scheduled.</p>
${strips(ASSETS)}
<h2>Piece counts are not yet reliable</h2>
<div class=flag><p>Reject count is derived as infeed minus good count rather than from scanned reject
reasons, and the reject-event log is empty across the entire history. Where the good-count sensor
registers more pieces than the infeed sensor saw, rejects go negative.</p>
<p>The net is unchanged at hour, shift and day level, so this is a measurement fault rather than a
timing artefact. Quality and OEE are withheld until the count inputs are reviewed.</p></div>`;
}

function linePage(a){
  const dr=D.filter(r=>r.asset===a&&inP(r.d)), s=agg(dr);
  if(!dr.length)return '<p class=empty>No records for this line in this period.</p>';
  const ar=A.filter(r=>r.asset===a&&inP(r.d));
  const by=new Map();
  ar.forEach(r=>{const k=r.state+'|'+r.impact,o=by.get(k)||{h:0,n:0};o.h+=r.h;o.n+=r.n;by.set(k,o)});
  const acts=[...by].map(([k,v])=>({st:k.split('|')[0],im:k.split('|')[1],...v}))
    .sort((x,y)=>y.h-x.h);
  const tot=acts.reduce((t,r)=>t+r.h,0)||1;
  const actRows=acts.map(r=>`<tr><th>${r.st.replace(/_/g,' ')}</th>
<td class=bar><i style="width:${(r.h/tot*100).toFixed(2)}%;background:var(--${IMP[r.im]||'nsc'})"></i></td>
<td>${r.im.replace(/_/g,' ')}</td><td>${hrs(r.h)}</td><td>${r.n.toLocaleString()}</td>
<td>${(r.h*60/r.n).toFixed(0)}</td></tr>`).join('');

  function reasons(imp,title,lead){
    const m=new Map();
    ar.filter(r=>r.impact===imp).forEach(r=>{const o=m.get(r.reason)||{h:0,n:0};o.h+=r.h;o.n+=r.n;m.set(r.reason,o)});
    const l=[...m].map(([k,v])=>({k,...v})).sort((x,y)=>y.h-x.h);
    if(!l.length)return '';
    const T=l.reduce((t,r)=>t+r.h,0),mx=l[0].h;let c=0;
    const rows=l.slice(0,20).map(r=>{c+=r.h;
      return `<tr><th>${r.k}</th><td class=bar><i style="width:${(r.h/mx*100).toFixed(2)}%;background:var(--${IMP[imp]})"></i></td>
<td>${hrs(r.h)}</td><td>${r.n.toLocaleString()}</td><td>${(r.h*60/r.n).toFixed(0)}</td>
<td>${(c/T*100).toFixed(0)}%</td></tr>`}).join('');
    return `<h2>${title}</h2><p class=note>${lead}</p>
<table><thead><tr><th>Reason</th><td class=bar></td><td>Hours</td><td>Events</td>
<td>Avg min</td><td>Cumulative</td></tr></thead><tbody>${rows}</tbody></table>`;
  }

  return `<h2>${a} at a glance</h2>
<table><thead><tr><td class=bar>Share of elapsed time</td><td>Available</td><td>Performance</td>
<td>Utilised</td><td>Scheduled h</td><td>Down h</td><td>Down events</td></tr></thead>
<tbody><tr><td class=bar>${stack(s)}</td><td>${pc(s.avail)}</td><td>${pc(s.perf)}</td>
<td>${pc(s.util)}</td><td>${hrs(s.sched)}</td><td>${hrs(s.unp)}</td>
<td>${ar.filter(r=>r.impact==='unplanned_stop').reduce((t,r)=>t+r.n,0).toLocaleString()}</td></tr>
</tbody></table>${KEY}
<h2>Hours by activity</h2>
<p class=note>What the line was doing, and whether it counts against availability.</p>
<table><thead><tr><th>Activity</th><td class=bar></td><td>Counts as</td><td>Hours</td>
<td>Events</td><td>Avg min</td></tr></thead><tbody>${actRows}</tbody></table>
${reasons('unplanned_stop','Why it stopped','Ranked by hours lost. Many short events point to a nuisance problem; few long ones point to a process or parts problem.')}
${reasons('planned_stop','Planned stops','Changeover, maintenance and setup — planned, but still time the line was not producing.')}
<h2>Availability, day by day</h2>${strips([a])}`;
}

function render(){
  document.getElementById('nav').innerHTML=
    [['overview','Overview'],...ASSETS.map(a=>[a,a])]
    .map(([k,l])=>`<button data-go="${k}"${k===page?' aria-current=page':''}>${l}</button>`).join('');
  document.getElementById('gran').innerHTML=
    [['day','Day'],['week','Week'],['month','Month'],['ytd','Year to date']]
    .map(([k,l])=>`<button data-g="${k}" aria-pressed="${k===gran}">${l}</button>`).join('');
  const ps=periodList(gran);
  if(!ps.includes(period))period=ps[0];
  document.getElementById('period').innerHTML=
    ps.map(p=>`<option value="${p}"${p===period?' selected':''}>${label(gran,p)}</option>`).join('');
  const ds=dates.filter(inP);
  document.getElementById('range').textContent=
    ds.length?ds.length+' production day'+(ds.length>1?'s':''):'no data';
  document.getElementById('sub').textContent=
    'Where the time went · '+label(gran,period)+' · four packaging lines';
  document.getElementById('main').innerHTML=page==='overview'?overview():linePage(page);
}
document.addEventListener('click',ev=>{
  const g=ev.target.closest('[data-g]'), p=ev.target.closest('[data-go]');
  if(g){gran=g.dataset.g;period=null;render()}
  else if(p){page=p.dataset.go;render()}
});
document.getElementById('period').addEventListener('change',e=>{period=e.target.value;render()});
render();
</script>
<style>.lnk{background:none;border:0;padding:0;font:inherit;font-weight:600;color:inherit;
text-decoration:underline;text-underline-offset:3px;cursor:pointer}</style>
</html>"""

OUT = Path("report"); OUT.mkdir(exist_ok=True)
html = PAGE.replace("__DATA__", _json.dumps(payload, separators=(",", ":")))
(OUT / "danbro.html").write_text(html, encoding="utf-8")
print("wrote report/danbro.html —", len(html)//1024, "KB")

wrote report/danbro.html — 572 KB


In [32]:
PJ = ["start_time","end_time","part","job","duration","run_time",
      "unplanned_stop_time","planned_stop_time","in_count","good_count",
      "ideal_time","ideal_cycle_time","target_count","target_cycles",
      "run_cycles","small_stops"]

runs = []
for ip, name in ASSETS.items():
    for ch in ("part","job"):
        try:
            d = get_events(ch, PJ, host=ip)
            d.insert(0,"grain",ch); d.insert(0,"asset",name)
            runs.append(d)
            print(f"{name:6} {ch:5} {len(d):5} runs")
        except Exception as ex:
            print(f"{name:6} {ch:5} FAILED {ex}")

rdf = pd.concat(runs, ignore_index=True)
numc = [c for c in PJ if c not in ("start_time","end_time","part","job")]
rdf[numc] = rdf[numc].astype(float)
rdf["start_time"] = pd.to_datetime(rdf["start_time"], utc=True)

p = rdf[rdf.grain=="part"].copy()
p["act_spp"] = p.run_time / p.good_count.replace(0, pd.NA)
p["std_spp"] = p.ideal_cycle_time
p["earned_h"] = p.ideal_time / 3600
p["actual_h"] = p.run_time / 3600

print("\nsample part runs:")
print(p[p.good_count>1000][["asset","part","job","actual_h","earned_h",
      "act_spp","std_spp","good_count"]].head(10).to_string(index=False))

FR8    part    300 runs
FR8    job     300 runs
AreaB  part     63 runs
AreaB  job      63 runs
AreaD  part     79 runs
AreaD  job      79 runs
AreaE  part     90 runs
AreaE  job      90 runs

sample part runs:
asset                    part     job  actual_h  earned_h   act_spp  std_spp  good_count
  FR8               ESTRADIOL 3044003 35.392685 12.861354  0.563857      NaN    225968.0
  FR8               ESTRADIOL 3044000 19.222824  6.638854  0.538878      NaN    128419.0
  FR8               ESTRADIOL 3043805 10.032803  2.689687  0.884879      NaN     40817.0
  FR8 CLEANSING_GELi_50G_j_US 3044085  4.029882  2.628519   1.14143      NaN     12710.0
  FR8                   ADVIL 3044039 12.809654  8.136970  0.497554      NaN     92683.0
  FR8                   ADVIL 3044039  0.527571  0.246515  0.316965      NaN      5992.0
  FR8                   ADVIL 3044039  1.948560  1.040758  0.417748      NaN     16792.0
  FR8               VOLTACARE 3044065 12.705246  8.742121  0.471929      NaN 

In [33]:
p["std_spp"] = p.ideal_time / p.good_count.replace(0, pd.NA)
p["eff"] = p.earned_h / p.actual_h

v = p[(p.good_count > 500) & (p.run_time > 0)].copy()
print("runs with meaningful volume:", len(v))
print(v[["actual_h","earned_h","eff","act_spp","std_spp"]].describe().to_string())

print("\nby part — actual vs standard hours:")
print(v.groupby("part").agg(
    runs=("eff","size"), actual_h=("actual_h","sum"), earned_h=("earned_h","sum"),
    std_spp=("std_spp","median"), act_spp=("act_spp","median"),
).assign(eff=lambda d: d.earned_h/d.actual_h)
 .sort_values("actual_h", ascending=False).head(12).to_string(
    float_format=lambda x: f"{x:,.3f}"))

runs with meaningful volume: 240
         actual_h    earned_h         eff
count  240.000000  240.000000  240.000000
mean    10.009545    5.575371    0.521918
std     13.335541    8.811217    0.226048
min      0.008176    0.000952    0.009976
25%      3.057279    1.354201    0.351847
50%      6.139426    2.564844    0.485959
75%     11.151332    5.992205    0.691008
max    105.365477   73.400187    1.701919

by part — actual vs standard hours:
                         runs  actual_h  earned_h std_spp act_spp   eff
part                                                                   
ESTRADIOL                  60   400.092   142.475   0.215   0.578 0.356
OMNARIS                    12   367.790   203.377   0.880   1.627 0.553
OTRIVIN                    13   302.316   205.897   0.883   2.134 0.681
OMNARIS_120P                8   171.009   111.894   1.260   1.783 0.654
ROGAINE_5b_TRIPLES         10   151.474    69.705   0.587   1.266 0.460
XHANCE_6k4ML                3   108.118    70.63

In [34]:
# the parts configuration lives in the dimensions endpoint
for path in ["dimensions/part", "parts", "configuration/parts"]:
    r = requests.get(f"http://{DEVICES[0]}/api/v0/{path}", timeout=10)
    print(r.status_code, path, "|", r.text[:400].replace("\n"," "))
    print("-"*60)

200 dimensions/part | {"data":{"values":[{"display_name":"No Part","is_in_configuration":true,"key":"null"},{"display_name":"Unknown Part","is_in_configuration":true,"key":"unknown"},{"display_name":"ABREVA, 3G","is_in_configuration":false,"key":"ABREVAi_3G"},{"display_name":"ADVIL, 2.3oz","is_in_configuration":false,"key":"ADVILi_2k3oz"},{"display_name":"ADVIL, 2.5oz","is_in_configuration":false,"key":"ADVILi_2k5oz"},
------------------------------------------------------------
404 parts | 
------------------------------------------------------------
404 configuration/parts | 
------------------------------------------------------------


In [35]:
p["ict_derived"] = p.ideal_time / p.run_cycles.replace(0, pd.NA)
chk = p[(p.run_cycles > 1000)].groupby("part").agg(
    runs=("ict_derived","size"),
    ict=("ict_derived","median"),
    ict_sd=("ict_derived","std"),
    spp_actual=("act_spp","median"),
).sort_values("runs", ascending=False)
print(chk.head(15).to_string(float_format=lambda x: f"{x:,.4f}"))

                         runs    ict  ict_sd spp_actual
part                                                   
ESTRADIOL                  63 0.3772  0.0010     0.6029
SOFDRA                     16 2.1607  0.0091     2.5823
OMNARIS                    13 0.9171  0.0084     1.6508
OTRIVIN                    12 1.5440  0.6696     2.1042
ROGAINE_5b_TRIPLES         10 0.6099  0.0137     1.2659
LOCACORTEN_VF_EARDROPS     10 2.5499  0.0243     3.6087
OMNARIS_120P                9 0.9446  0.4747     1.7325
ABREVAi_3G                  8 0.6020  0.0407     0.7417
ADVIL                       8 0.5477  0.0010     0.5941
ROGAINE_5b_SINGLES          6 0.6211  0.0036     2.6456
PREMARINi_30G               6 0.7582  0.0027     1.3269
CLEANSING_GELi_50G_j_US     6 0.6677  0.0010     1.0249
REGAINE_5b_60ML_TRIPLES     6 0.6152  0.0058     1.1380
ROGAINE_2b_SINGLES          5 0.6285  0.0186     2.9784
OTRIVIN_PUMPi_20ML          4 0.8912  0.0059     1.1789


In [44]:
import json as _json
from pathlib import Path

OUT = Path("report"); OUT.mkdir(exist_ok=True)
TZ = "America/Toronto"

# ---------- hourly, with product ----------
hp = w.copy()
hp["sched"] = hp.run_time + hp.unplanned_stop_time + hp.planned_stop_time
hp = hp[hp.sched > 0].copy()
lt = hp.start_time.dt.tz_convert(TZ)

ht = pd.DataFrame({
    "a": hp["asset"].values, "d": lt.dt.strftime("%Y-%m-%d").values,
    "hh": lt.dt.strftime("%H:%M").values, "sh": hp["shift"].values,
    "p": hp["part"].values, "j": hp["job"].values,
    "r": (hp["run_time"]/60).round(1).values,
    "u": (hp["unplanned_stop_time"]/60).round(1).values,
    "pl": (hp["planned_stop_time"]/60).round(1).values,
    "i": (hp["ideal_time"]/60).round(2).values,
    "g": hp["good_count"].round(0).values,
})
hours = ht.to_dict("records")
print("hours:", len(hours))

# ---------- standards from run-cycle ideal ----------
p = rdf[rdf.grain == "part"].copy()
p["ict"] = p.ideal_time / p.run_cycles.replace(0, pd.NA)
lt2 = p.start_time.dt.tz_convert(TZ)
p["d"] = lt2.dt.strftime("%Y-%m-%d")

std = (p[p.run_cycles > 200].groupby("part")
        .agg(ict=("ict","median"), sd=("ict","std"), n=("ict","size"))
        .reset_index().round(4))
std["sd"] = std.sd.fillna(0)

import numpy as np

def runs_payload(g):
    x = rdf[rdf["grain"] == g].copy()
    x["ict"] = x["ideal_time"] / x["run_cycles"].replace(0, np.nan)
    x = x[x["run_time"] > 0]
    lx = x["start_time"].dt.tz_convert(TZ)
    out = pd.DataFrame({
        "a": x["asset"].values, "d": lx.dt.strftime("%Y-%m-%d").values,
        "p": x["part"].values, "j": x["job"].values,
        "ah": (x["run_time"]/3600).round(3).values,
        "eh": (x["ideal_time"]/3600).round(3).values,
        "g": x["good_count"].round(0).values,
        "ict": x["ict"].round(4).values,
        "dn": ((x["unplanned_stop_time"] + x["planned_stop_time"])/3600).round(2).values,
    })
    return out.where(pd.notna(out), None).to_dict("records")

# ---------- optional SAP csv ----------
sap = {}
csv = Path("sap_standards.csv")
if csv.exists():
    s = pd.read_csv(csv)
    s.columns = [c.strip().lower() for c in s.columns]
    k, v = s.columns[0], s.columns[1]
    sap = {str(r[k]): float(r[v]) for _, r in s.iterrows() if pd.notna(r[v])}
    print(f"loaded {len(sap)} SAP standards from {csv}")

payload = {
    "assets": sorted(ASSETS.values()),
    "daily": daily.to_dict("records"),
    "activity": slim.round(4).to_dict("records"),
    "hours": hours,
    "std": std.to_dict("records"),
    "partruns": runs_payload("part"),
    "jobruns": runs_payload("job"),
}
print({k: (len(v) if isinstance(v, list) else v) for k, v in payload.items()})

ADD = r"""
/* ---- products ---- */
.tools{display:flex;gap:12px;align-items:center;flex-wrap:wrap;margin:14px 0 18px;font-size:14px}
.tools button,.tools label{font:inherit;font-size:14px;border:1px solid var(--rule);
background:#fff;padding:7px 12px;cursor:pointer;color:inherit}
.tools button:hover,.tools label:hover{background:var(--panel)}
.tools input[type=file]{display:none}
td input.sap{width:92px;font:inherit;font-size:14px;text-align:right;
border:1px solid var(--rule);padding:5px 7px;background:#FFFDF5}
td input.sap:focus{outline:2px solid var(--ink);outline-offset:0}
.over{color:var(--unp)}.under{color:var(--run)}
.scroll{max-height:560px;overflow:auto;border-bottom:1px solid var(--rule)}
.scroll thead th,.scroll thead td{position:sticky;top:0;background:#fff;z-index:1}
small.warn{display:block;color:var(--unp);font-size:13px;margin-top:6px}
"""

JS = r"""
const H=DATA.hours, STD=DATA.std, PR=DATA.partruns, JR=DATA.jobruns;
let sap=Object.assign({},__SAP__), unit='spp';
const U={spp:['Seconds per piece',v=>v],
         pph:['Pieces per hour',v=>v?3600/v:0],
         h1000:['Hours per 1,000',v=>v*3.6]};
const toSpp=v=>U[unit][1](v);
const n3=v=>v==null||!isFinite(v)?'—':v.toFixed(3);

function prodPage(){
  const pr=PR.filter(r=>inP(r.d));
  if(!pr.length)return '<p class=empty>No part runs in this period.</p>';
  const m=new Map();
  pr.forEach(r=>{const o=m.get(r.p)||{ah:0,eh:0,g:0,n:0,dn:0};
    o.ah+=r.ah;o.eh+=r.eh;o.g+=r.g;o.dn+=r.dn;o.n++;m.set(r.p,o)});
  const ictOf={};STD.forEach(s=>ictOf[s.part]=s);
  const rows=[...m].map(([k,v])=>{
    const s=ictOf[k]||{}, act=v.g>0?v.ah*3600/v.g:null,
          sp=sap[k]!=null?toSpp(sap[k]):null;
    return {k,...v,ict:s.ict,sd:s.sd,act,sp,
      sapEarned:sp!=null?v.g*sp/3600:null}})
    .sort((x,y)=>y.ah-x.ah);
  const cell=(a,b)=>{if(a==null||b==null||!b)return '<td>—</td>';
    const r=a/b;return `<td class="${r>1.15?'over':r<0.95?'under':''}">${r.toFixed(2)}×</td>`};
  const body=rows.map(r=>`<tr><th>${r.k}</th>
<td>${n3(r.ict)}${r.sd>0.05?'<small class=warn>unstable</small>':''}</td>
<td><input class=sap data-p="${r.k}" value="${sap[r.k]!=null?sap[r.k]:''}" placeholder="—"></td>
<td>${n3(r.act)}</td>${cell(r.act,r.ict)}${cell(r.act,r.sp)}
<td>${hrs(r.ah)}</td><td>${hrs(r.eh)}</td>
<td>${r.sapEarned!=null?hrs(r.sapEarned):'—'}</td>
<td>${r.sapEarned!=null?hrs(r.ah-r.sapEarned):'—'}</td><td>${r.n}</td></tr>`).join('');
  const opts=Object.entries(U).map(([k,v])=>
    `<option value="${k}"${k===unit?' selected':''}>${v[0]}</option>`).join('');
  return `<h2>Standards and variance</h2>
<p class=note>Vorne's ideal cycle time is recovered from run-cycle data and matches the device
configuration to within half a percent. The SAP column is yours to fill — paste a file or type
values in, and pick the unit they are in.</p>
<div class=tools>
<label for=sapfile>Load CSV</label><input type=file id=sapfile accept=".csv,text/csv">
<button id=sapexp>Export CSV</button><button id=sapclr>Clear SAP values</button>
<span>Unit of SAP values</span><select id=sapunit>${opts}</select></div>
<div class=scroll><table><thead><tr><th>Part</th><td>Vorne ICT</td><td>SAP standard</td>
<td>Actual s/pc</td><td>vs Vorne</td><td>vs SAP</td><td>Actual h</td><td>Earned h, Vorne</td>
<td>Earned h, SAP</td><td>Variance h</td><td>Runs</td></tr></thead><tbody>${body}</tbody></table></div>
<small class=warn>Seconds per piece depends on good count, which is not yet reliable on these
devices. Treat the ratios as directional until the count inputs are reviewed.</small>
${runTable(pr,'Part runs','One row per part run, longest first.')}
${runTable(JR.filter(r=>inP(r.d)),'Job runs','Same runs grouped by job or lot number.')}`;
}

function runTable(rs,title,lead){
  if(!rs.length)return '';
  const l=[...rs].sort((x,y)=>y.ah-x.ah).slice(0,60);
  const body=l.map(r=>{const act=r.g>0?r.ah*3600/r.g:null, v=act&&r.ict?act/r.ict:null;
    return `<tr><th>${r.p}</th><td>${r.j||'—'}</td><td>${r.a}</td><td>${nice(r.d)}</td>
<td>${hrs(r.ah)}</td><td>${hrs(r.eh)}</td><td>${hrs(r.dn)}</td>
<td>${r.g.toLocaleString()}</td><td>${n3(act)}</td>
<td class="${v>1.15?'over':v&&v<0.95?'under':''}">${v?v.toFixed(2)+'×':'—'}</td></tr>`}).join('');
  return `<h2>${title}</h2><p class=note>${lead}</p>
<div class=scroll><table><thead><tr><th>Part</th><td>Job</td><td>Line</td><td>Started</td>
<td>Actual h</td><td>Earned h</td><td>Stopped h</td><td>Good</td><td>s/pc</td>
<td>vs Vorne</td></tr></thead><tbody>${body}</tbody></table></div>`;
}

function hourTable(a){
  const rs=H.filter(r=>r.a===a&&inP(r.d)).sort((x,y)=>x.d+x.hh<y.d+y.hh?1:-1).slice(0,400);
  if(!rs.length)return '';
  const body=rs.map(r=>`<tr><th>${nice(r.d)}</th><td>${r.hh}</td><td>${r.sh?r.sh.replace(/_/g,' '):'—'}</td>
<td>${r.p||'—'}</td><td>${r.j||'—'}</td><td>${r.r.toFixed(0)}</td><td>${r.u.toFixed(0)}</td>
<td>${r.pl.toFixed(0)}</td><td>${r.g.toLocaleString()}</td>
<td>${r.r>0?((r.i/r.r)*100).toFixed(0)+'%':'—'}</td></tr>`).join('');
  return `<h2>Hour by hour</h2>
<p class=note>What ran in each scheduled hour, newest first. Performance is ideal time over run time
for that hour alone.</p>
<div class=scroll><table><thead><tr><th>Day</th><td>Hour</td><td>Shift</td><td>Product</td>
<td>Job</td><td>Run min</td><td>Down min</td><td>Planned min</td><td>Good</td>
<td>Perf</td></tr></thead><tbody>${body}</tbody></table></div>`;
}

function csvOut(){
  const l=['part,value,unit'];
  Object.keys(sap).sort().forEach(k=>l.push(`"${k}",${sap[k]},${unit}`));
  const b=new Blob([l.join('\n')],{type:'text/csv'}), u=URL.createObjectURL(b);
  const a=document.createElement('a');a.href=u;a.download='sap_standards.csv';a.click();
  URL.revokeObjectURL(u);
}
document.addEventListener('input',ev=>{
  const i=ev.target.closest('input.sap');if(!i)return;
  const v=parseFloat(i.value);
  if(isFinite(v)&&v>0)sap[i.dataset.p]=v;else delete sap[i.dataset.p];
  const act=i.closest('tr').children[3].textContent;
  render();setTimeout(()=>{const n=document.querySelector(`input.sap[data-p="${i.dataset.p}"]`);
    if(n){n.focus();n.setSelectionRange(n.value.length,n.value.length)}},0);
});
document.addEventListener('change',ev=>{
  if(ev.target.id==='sapunit'){unit=ev.target.value;render()}
  if(ev.target.id==='sapfile'){const f=ev.target.files[0];if(!f)return;
    const fr=new FileReader();
    fr.onload=()=>{fr.result.split(/\r?\n/).slice(1).forEach(l=>{
      const c=l.match(/("[^"]*"|[^,]+)/g);if(!c||c.length<2)return;
      const k=c[0].replace(/"/g,'').trim(), v=parseFloat(c[1]);
      if(k&&isFinite(v))sap[k]=v});render()};
    fr.readAsText(f)}
});
document.addEventListener('click',ev=>{
  if(ev.target.id==='sapexp')csvOut();
  if(ev.target.id==='sapclr'){sap={};render()}
});
"""

base = PAGE                      # always start fresh
base = base.replace("</style>", ADD + "</style>", 1)
base = base.replace("render();\n</script>", JS + "\nrender();\n</script>", 1)
base = base.replace("[['overview','Overview'],...ASSETS.map(a=>[a,a])]",
                    "[['overview','Overview'],...ASSETS.map(a=>[a,a]),['products','Products']]", 1)
base = base.replace("page==='overview'?overview():linePage(page)",
                    "page==='overview'?overview():page==='products'?prodPage():linePage(page)", 1)
base = base.replace("<h2>Availability, day by day</h2>${strips([a])}`;",
                    "<h2>Availability, day by day</h2>${strips([a])}${hourTable(a)}`;", 1)

# sanity: each must appear exactly once
for probe in ["function prodPage", "const H=DATA.hours", "const DATA=__DATA__"]:
    assert base.count(probe) == 1, f"{probe} appears {base.count(probe)}×"

html = (base.replace("__DATA__", _json.dumps(clean(payload), allow_nan=False,
                                             separators=(",", ":")))
            .replace("__SAP__", _json.dumps(sap)))
(OUT / "danbro.html").write_text(html, encoding="utf-8")
print("wrote —", len(html)//1024, "KB")

hours: 7392
{'assets': 4, 'daily': 649, 'activity': 4323, 'hours': 7392, 'std': 39, 'partruns': 379, 'jobruns': 379}
wrote — 1680 KB


In [ ]:
src = (OUT/"danbro.html").read_text(encoding="utf-8")
for probe in ["function prodPage", "function hourTable", "const H=DATA.hours",
              "['products','Products']", "page==='products'",
              "const DATA=__DATA__", "__SAP__", "NaN"]:
    print(f"{probe:28} {src.count(probe)}")

wrote — 1681 KB | NaN count: 0


In [42]:
# is the data actually in the file?
src = (OUT/"danbro.html").read_text(encoding="utf-8")
i = src.find("const DATA=")
print(src[i:i+300])
print("...")
j = src.find("render();\n</script>")
print(src[j-400:j+30] if j>0 else "render() call NOT FOUND")

const DATA={"assets":["AreaB","AreaD","AreaE","FR8"],"daily":[{"asset":"AreaB","d":"2026-03-25","run":0.0,"unp":0.1645,"pln":0.0,"nsc":14.8287,"ideal":0.0},{"asset":"AreaB","d":"2026-03-26","run":0.0,"unp":0.0,"pln":0.0,"nsc":24.0,"ideal":0.0},{"asset":"AreaB","d":"2026-03-27","run":0.0,"unp":0.0,"p
...
   fr.onload=()=>{fr.result.split(/\r?\n/).slice(1).forEach(l=>{
      const c=l.match(/("[^"]*"|[^,]+)/g);if(!c||c.length<2)return;
      const k=c[0].replace(/"/g,'').trim(), v=parseFloat(c[1]);
      if(k&&isFinite(v))sap[k]=v});render()};
    fr.readAsText(f)}
});
document.addEventListener('click',ev=>{
  if(ev.target.id==='sapexp')csvOut();
  if(ev.target.id==='sapclr'){sap={};render()}
});

render();
</script>
<style>.ln


In [43]:
base.replace("render();\n</script>", JS + "\nrender();\n</script>")

'<!doctype html><html lang=en><meta charset=utf-8>\n<meta name=viewport content="width=device-width,initial-scale=1">\n<title>Danbro packaging</title>\n<link href="https://fonts.googleapis.com/css2?family=Libre+Franklin:wght@400;600;800&display=swap" rel=stylesheet>\n<style>\n:root{--ink:#15191B;--mute:#5A6368;--rule:#DADFE1;--panel:#F4F6F6;\n--run:#1F6F5C;--unp:#B3261E;--pln:#C98A0B;--nsc:#B9C0C4}\n*{box-sizing:border-box}\nbody{margin:0;background:#fff;color:var(--ink);\nfont:400 16px/1.5 \'Libre Franklin\',system-ui,sans-serif;font-variant-numeric:tabular-nums}\n.wrap{max-width:1080px;margin:0 auto;padding:48px 24px 96px}\nh1{font-weight:800;font-size:clamp(26px,4vw,40px);letter-spacing:-.02em;margin:0 0 4px}\n.sub{color:var(--mute);margin:0 0 28px}\nnav{display:flex;gap:4px;flex-wrap:wrap;border-bottom:1px solid var(--rule);margin-bottom:24px}\nnav button{background:none;border:0;border-bottom:2px solid transparent;margin-bottom:-1px;\npadding:9px 12px;font:inherit;font-weight:600;